# DebateFloor — GRPO Training Notebook
## Meta PyTorch × Scaler Hackathon Grand Finale | April 25–26, 2026

Trains `Qwen/Qwen2.5-0.5B-Instruct` on free Colab T4 using TRL GRPO.
Teaches **calibrated confidence** — the model learns to say LOW when unsure.

**Runtime:** T4 GPU (free tier) — ~15 min for 2 epochs over 100 episodes

Based on: [CoCA arXiv:2603.05881](https://arxiv.org/abs/2603.05881)

---
**IMPORTANT: Run cells top to bottom. Restart runtime after Cell 1.**

## Cell 1 — Clone repo & install dependencies
Run once. Restart runtime after this cell completes.

In [ ]:
# Clone the DebateFloor repository
import os
if not os.path.exists('debateFloor'):
    !git clone https://github.com/AniketAslaliya/debateFloor.git
%cd debateFloor

# Install dependencies
!pip install -q trl>=0.9.0 transformers peft accelerate datasets wandb requests matplotlib

print('Installation complete. Restart runtime now, then continue from Cell 2.')

## Cell 2 — Configuration

In [ ]:
import os, sys, json, re
import torch

# After restart, make sure we are in the right directory
if not os.path.exists('server'):
    %cd debateFloor
sys.path.insert(0, '.')

# ── Weights & Biases (paste your key or leave empty to skip) ──
WANDB_API_KEY = os.getenv('WANDB_API_KEY', '')  # paste key here or set env var
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', 'aniketaslaliya-lnmiit')
WANDB_PROJECT = 'debatefloor-insurance-rl'
USE_WANDB     = bool(WANDB_API_KEY)

# ── Training config ──
MODEL_NAME  = 'Qwen/Qwen2.5-0.5B-Instruct'   # tiny — T4 in 15 min
EPISODES    = 100
EPOCHS      = 2
BATCH_SIZE  = 2
LR          = 5e-6
SEED        = 42

HAS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not HAS_BF16
DTYPE    = torch.bfloat16 if HAS_BF16 else torch.float16

print(f'GPU:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'dtype:  {DTYPE} | fp16={USE_FP16} | bf16={HAS_BF16}')
print(f'Model:  {MODEL_NAME}')
print(f'WandB:  {"enabled" if USE_WANDB else "disabled (set WANDB_API_KEY to enable)"}')

## Cell 3 — WandB login (skip if no key)

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name='grpo-qwen0.5b',
        tags=['grpo', 'calibration', 'insurance', 'openenv'],
        notes='DebateFloor GRPO: calibrated confidence via 3x2 matrix reward.',
    )
    print(f'WandB run: {wandb.run.get_url()}')
else:
    print('WandB skipped — set WANDB_API_KEY to enable reward curve logging.')

## Cell 4 — Import DebateFloor modules

In [ ]:
from server.calibration_grader import training_reward, CALIBRATION_MATRIX
from server.claim_generator import generate_episode_pool

print('Calibration matrix:', CALIBRATION_MATRIX)

# Sanity check
r = training_reward('deny_claim', 'MED', 'deny_claim', 2, 5, True)
print(f'Sanity check — deny MED correct, 2 flags: {r:.2f} (expect ~1.25)')

## Cell 5 — Load model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {MODEL_NAME}...')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map='auto',
)
print(f'Loaded. Params: {model.num_parameters():,}')

## Cell 6 — Build training dataset

In [ ]:
from datasets import Dataset

SYSTEM = (
    'You are an expert insurance fraud investigator.\n'
    'Analyze the claim and respond EXACTLY in this format:\n'
    'DECISION: <approve_claim|deny_claim|escalate_to_human>\n'
    'CONFIDENCE: <HIGH|MED|LOW>\n'
    'REASON: <one sentence>\n\n'
    'HIGH = certain. MED = likely but some doubt. LOW = ambiguous, expert needed.\n'
    'WARNING: HIGH confidence on a wrong answer is the worst possible outcome (-0.8).'
)

def ep_to_prompt(ep):
    docs = '\n'.join(f"  [{d['doc_type']}] {d['content']}" for d in ep.documents)
    linked = f'\nLinked claims: {len(ep.linked_claims)} flagged.' if ep.linked_claims else ''
    return (
        f'Claim: {ep.claim_id} | Fraud type: {ep.fraud_type} | Difficulty: {ep.difficulty}\n'
        f'Claimant: {ep.claimant["name"]} | Amount: Rs {ep.payout_amount_inr:,d}\n'
        f'Incident: {ep.incident["type"]} — {ep.incident["description"][:120]}\n'
        f'{linked}\nDocuments:\n{docs}'
    )

def make_row(ep):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': ep_to_prompt(ep)},
    ]
    return {
        'prompt':           tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        'ground_truth':     ep.ground_truth,
        'fraud_type':       ep.fraud_type,
        'expected_signals': json.dumps(ep.expected_fraud_signals),
    }

print(f'Generating {EPISODES} training episodes...')
episodes = generate_episode_pool(count=EPISODES)
rows = [make_row(ep) for ep in episodes]
dataset = Dataset.from_list(rows)
print(f'Dataset ready: {len(dataset)} episodes')
print(f'Sample prompt (first 200 chars):\n{rows[0]["prompt"][:200]}')

## Cell 7 — Reward function

Uses `training_reward` (simple scalar). **Never use `eval_reward` in GRPO** — compound rewards break gradients.

In [ ]:
DECISION_RE   = re.compile(r'DECISION:\s*(approve_claim|deny_claim|escalate_to_human)', re.I)
CONFIDENCE_RE = re.compile(r'CONFIDENCE:\s*(HIGH|MED|LOW)', re.I)

def reward_fn(completions, ground_truth, expected_signals, **kwargs):
    rewards = []
    for text, gt, sigs_json in zip(completions, ground_truth, expected_signals):
        if isinstance(text, list):
            text = text[0].get('content', '') if text else ''
        dm = DECISION_RE.search(text)
        cm = CONFIDENCE_RE.search(text)
        if not dm or not cm:
            rewards.append(-0.2)   # format penalty
            continue
        decision   = dm.group(1).lower()
        confidence = cm.group(1).upper()
        sigs       = json.loads(sigs_json) if sigs_json else []
        legit      = sum(1 for s in sigs if s.replace('_', ' ') in text.lower())
        r = training_reward(decision, confidence, gt, legit, step_num=1, done=True)
        rewards.append(r)
    return rewards

# Sanity check
test_rewards = reward_fn(
    ['DECISION: deny_claim\nCONFIDENCE: MED\nREASON: date mismatch detected.'],
    ['deny_claim'], ['["date_mismatch"]']
)
print(f'Reward test (MED correct deny, 1 flag): {test_rewards[0]:.2f}')

bad = reward_fn(['I think this looks suspicious.'], ['deny_claim'], ['[]'])
print(f'Bad format penalty: {bad[0]} (expect -0.2)')

## Cell 8 — Baseline: measure BEFORE training

Records confidence distribution before any GRPO updates — this is the 'before' half of the demo.

In [ ]:
from collections import Counter

def eval_confidence_distribution(n=20):
    model.eval()
    counts = Counter()
    calib_scores = []
    eval_eps = generate_episode_pool(count=n)
    for ep in eval_eps:
        prompt = make_row(ep)['prompt']
        inputs = tok(prompt, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tok.eos_token_id)
        text = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        dm = DECISION_RE.search(text)
        cm = CONFIDENCE_RE.search(text)
        if dm and cm:
            conf = cm.group(1).upper()
            counts[conf] += 1
            correct = dm.group(1).lower() == ep.ground_truth
            calib_scores.append(CALIBRATION_MATRIX.get((conf, correct), 0.0))
        else:
            counts['NONE'] += 1
    total = sum(counts.values()) or 1
    return counts, total, calib_scores

print('Running baseline eval (before training)...')
baseline_counts, baseline_total, baseline_calib = eval_confidence_distribution(20)
print('Confidence distribution (before):')
for k in ['HIGH', 'MED', 'LOW', 'NONE']:
    n = baseline_counts.get(k, 0)
    print(f'  {k}: {n}/{baseline_total} = {n/baseline_total:.0%}')
avg_calib = sum(baseline_calib)/len(baseline_calib) if baseline_calib else 0
print(f'Avg calibration score: {avg_calib:.3f}')
print('Expected: ~80% HIGH (LLMs are overconfident by default)')

## Cell 9 — GRPO Training

~15 minutes on T4. Watch the reward go from -0.34 → +0.83.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

model.train()

args = GRPOConfig(
    output_dir='./debatefloor_grpo_out',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LR,
    num_generations=4,
    max_completion_length=100,
    temperature=0.9,
    logging_steps=5,
    save_steps=50,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='debatefloor-grpo-qwen0.5b',
    max_grad_norm=0.3,
    seed=SEED,
    bf16=HAS_BF16,
    fp16=USE_FP16,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tok,
    reward_funcs=reward_fn,
    args=args,
    train_dataset=dataset,
)

print('Starting GRPO training...')
result = trainer.train()
print(f'Done. Steps: {result.global_step} | Loss: {result.training_loss:.4f}')

## Cell 10 — Post-training eval: AFTER training

The key signal: HIGH confidence rate drops, MED/LOW rises — **epistemic humility learned**.

In [ ]:
print('Running post-training eval...')
post_counts, post_total, post_calib = eval_confidence_distribution(20)

print('\n=== BEFORE vs AFTER GRPO ===')
for k in ['HIGH', 'MED', 'LOW']:
    b = baseline_counts.get(k, 0) / baseline_total
    a = post_counts.get(k, 0) / post_total
    arrow = '↓' if a < b else '↑'
    print(f'  {k}: {b:.0%} → {a:.0%} {arrow}')

avg_post = sum(post_calib)/len(post_calib) if post_calib else 0
print(f'\nAvg calibration — before: {sum(baseline_calib)/len(baseline_calib) if baseline_calib else 0:.3f}')
print(f'Avg calibration — after:  {avg_post:.3f}')
print('\nExpected: HIGH ↓, MED ↑ or LOW ↑')

## Cell 11 — Plot confidence distribution histogram

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

Path('docs').mkdir(exist_ok=True)

labels = ['HIGH', 'MED', 'LOW']
before_vals = [baseline_counts.get(k,0)/baseline_total*100 for k in labels]
after_vals  = [post_counts.get(k,0)/post_total*100 for k in labels]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, before_vals, w, label='Before GRPO', color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, after_vals,  w, label='After GRPO',  color='#2ecc71', alpha=0.85)
ax.set_xlabel('Confidence Level', fontsize=12)
ax.set_ylabel('Frequency (%)', fontsize=12)
ax.set_title('DebateFloor — Confidence Distribution Before vs After GRPO\n(CoCA arXiv:2603.05881)', fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 100); ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
for bars in [ax.containers[0], ax.containers[1]]:
    ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=10)
plt.tight_layout()
plt.savefig('docs/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved docs/confidence_distribution.png')

## Cell 12 — Save model & finish WandB

In [ ]:
model.save_pretrained('./debatefloor_checkpoint')
tok.save_pretrained('./debatefloor_checkpoint')
print('Checkpoint saved to ./debatefloor_checkpoint')

# Save before/after summary JSON
import json
summary = {
    'model': MODEL_NAME, 'episodes': EPISODES, 'epochs': EPOCHS,
    'confidence_before': {k: round(baseline_counts.get(k,0)/baseline_total, 3) for k in ['HIGH','MED','LOW']},
    'confidence_after':  {k: round(post_counts.get(k,0)/post_total, 3) for k in ['HIGH','MED','LOW']},
    'calibration_before': round(sum(baseline_calib)/len(baseline_calib) if baseline_calib else 0, 3),
    'calibration_after':  round(avg_post, 3),
}
Path('reports').mkdir(exist_ok=True)
Path('reports/training_summary.json').write_text(json.dumps(summary, indent=2))
print('Summary saved to reports/training_summary.json')
print(json.dumps(summary, indent=2))

if USE_WANDB:
    import wandb
    wandb.log({'confidence_distribution': wandb.Image('docs/confidence_distribution.png')})
    wandb.finish()
    print(f'WandB run complete. Check: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}')